# Tarea 1 · Parte 2 — Del LLM al clasificador

**Flujo:**
1. Partición estratificada 70/15/15 (`random_state=42`) **antes de todo**, para que el test quede aislado.
2. Línea base *zero-shot* (sin entrenamiento) sobre 40 titulares del test.
3. Supervisión débil: un LLM (OpenAI o Gemini) etiqueta todo el dataset → `datos/etiquetas_llm.csv` + auditoría humana de 20 casos.
4. *Fine-tuning* de **dos modelos** (DistilBERT multilingüe y BETO) con las etiquetas del LLM; se elige el mejor en validación.
5. Evaluación final en el **test reservado** contra la etiqueta de referencia (`etiqueta`, derivada de la sección del diario).
6. Comparativa y decisión.


### Paso 0 · Preparar el entorno
**Objetivo:** confirmar que las librerías están instaladas y ver si hay GPU.
**Qué hace:** muestra la versión de `transformers` (ModernBERT requiere ≥ 4.48) y si hay GPU. Sin GPU el
entrenamiento usa la CPU, que es más lento pero suficiente para ~300 titulares cortos. La clave de OpenAI se lee del
archivo `.env`, que no se sube a GitHub.
**Resultado esperado:** versión de transformers ≥ 4.48 y el dispositivo que se usará.

In [1]:
import os
import torch, transformers
print("transformers:", transformers.__version__)
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (sin GPU)")
print("datos/:", os.listdir("datos"))

c:\Users\valer\anaconda3\envs\webscraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.20.1
Dispositivo: CPU (sin GPU)
datos/: ['etiquetas_llm.csv', 'noticias.csv', 'particion.csv', 'raw']


Paso 1 · Configuración y carga de datos
Objetivo: cargar datos/noticias.csv (salida de la Parte 1) y fijar la semilla para que los resultados se repitan.

Qué hace: importa librerías, fija SEMILLA = 42, define las 4 categorías y su código numérico (LABEL2ID / ID2LABEL, que el modelo necesita, como en la clase 2) y muestra cuántas noticias hay por categoría.

Resultado esperado: (433, 7) y entre 100 y 115 noticias por categoría.

In [2]:
import json, time, random
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEMILLA = 42                                         # misma semilla = mismos resultados
random.seed(SEMILLA); np.random.seed(SEMILLA)

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
LABEL2ID = {c: i for i, c in enumerate(CATEGORIAS)}  # texto → número   ('politica' → 0)
ID2LABEL = {i: c for c, i in LABEL2ID.items()}       # número → texto   (0 → 'politica')
os.makedirs("modelo", exist_ok=True)
os.makedirs("figuras", exist_ok=True)

df = pd.read_csv("datos/noticias.csv", encoding="utf-8-sig")
df["titular"] = df["titular"].astype(str).str.strip()
print(df.shape)
df["etiqueta"].value_counts()

(433, 7)


etiqueta
politica              115
deportes              110
economia              108
ciencia_tecnologia    100
Name: count, dtype: int64

### Paso 2 · Partición 70/15/15 estratificada
**Objetivo:** separar entrenamiento (70 %), validación (15 %) y prueba (15 %) **antes de cualquier otra cosa**.

**Qué hace:** `train_test_split` se aplica dos veces (70/30, y luego el 30 se divide en dos mitades). `stratify`
mantiene la misma proporción de cada categoría en las 3 partes. La partición se guarda en `datos/particion.csv`.

**Por qué primero:** el **test** queda aislado. No se usa para entrenar ni para elegir modelo, y así la nota final
es honesta (clase 2: entrenar → testear → *deployment*).

<!-- **Resultado esperado:** ≈ 303 / 65 / 65 noticias, con las 4 categorías repartidas de forma pareja -->

In [3]:
# 1) 70 % train y 30 % temporal
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=SEMILLA)
# 2) el 30 % temporal se divide en mitades: 15 % validación y 15 % test
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["etiqueta"], random_state=SEMILLA)

for nombre, parte in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df.loc[parte.index, "particion"] = nombre
df[["url", "titular", "etiqueta", "particion"]].to_csv("datos/particion.csv", index=False, encoding="utf-8-sig")

resumen = pd.crosstab(df["etiqueta"], df["particion"])[["train", "val", "test"]]
resumen.loc["TOTAL"] = resumen.sum()
print(f"train={len(train_df)} | val={len(val_df)} | test={len(test_df)}")
resumen

train=303 | val=65 | test=65


particion,train,val,test
etiqueta,,,
ciencia_tecnologia,70,15,15
deportes,77,16,17
economia,76,16,16
politica,80,18,17
TOTAL,303,65,65


**Resultado:** 303 titulares para entrenar, 65 para validar y 65 para el test final. Cada categoría mantiene su
proporción en las tres partes (por ejemplo, ciencia y tecnología: 70 / 15 / 15), así ninguna parte queda sesgada.

### Paso 3 · Línea base *zero-shot* (sin entrenamiento)
**Objetivo:** medir cuánto acierta un modelo que **no fue entrenado** con nuestros datos. Es el "piso" a superar.

**Qué hace:** usa `pipeline("zero-shot-classification")` (visto en *lecture-os-models*) con un modelo NLI en español,
`Recognai/bert-base-spanish-wwm-cased-xnli`. Recibe las 4 categorías en palabras ("política", "economía"...) y elige
la que mejor completa la frase "Esta noticia trata sobre {}.". Se evalúa en **40 titulares del test**
(10 por categoría). Solo se predice: no se entrena nada con el test.

**Resultado esperado:** accuracy y F1 macro del zero-shot, y el reporte por categoría.

In [4]:
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1          # 0 = GPU, -1 = CPU
zs = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli", device=DEVICE)
NOMBRES_ZS = {"política": "politica", "economía": "economia",
              "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}

muestra_zs = test_df.groupby("etiqueta").sample(n=10, random_state=SEMILLA).copy()   # 10 por categoría
t0 = time.time()
salidas = zs(muestra_zs["titular"].tolist(), candidate_labels=list(NOMBRES_ZS),
             hypothesis_template="Esta noticia trata sobre {}.")
lat_zs = (time.time() - t0) / len(muestra_zs)                       # segundos por titular
muestra_zs["pred_zs"] = [NOMBRES_ZS[s["labels"][0]] for s in salidas]   # la etiqueta con mayor puntaje

acc_zs = accuracy_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"])
f1_zs = f1_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"], average="macro")
print(f"Zero-shot ({len(muestra_zs)} titulares) → Accuracy: {acc_zs:.3f} | F1 macro: {f1_zs:.3f}")
print(classification_report(muestra_zs["etiqueta"], muestra_zs["pred_zs"], zero_division=0))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 333.83it/s]


Zero-shot (40 titulares) → Accuracy: 0.575 | F1 macro: 0.542
                    precision    recall  f1-score   support

ciencia_tecnologia       0.67      0.20      0.31        10
          deportes       0.46      0.60      0.52        10
          economia       0.83      0.50      0.62        10
          politica       0.56      1.00      0.71        10

          accuracy                           0.57        40
         macro avg       0.63      0.57      0.54        40
      weighted avg       0.63      0.57      0.54        40



**Resultado del zero-shot:** F1 macro de **0.54** (accuracy 0.58) en 40 titulares del test, sin ningún entrenamiento.
Este es el piso a superar. El modelo tiende a clasificar casi todo como **política**: recall de 1.00, pero precisión
de solo 0.56, es decir, también envía ahí titulares de otras categorías. Le cuesta mucho **ciencia y tecnología**
(recall de 0.20): solo reconoce 2 de cada 10, porque términos como "app", "iPhone" o "inteligencia artificial" no se
asocian fácilmente a la frase "ciencia y tecnología" sin ejemplos.

### Paso 4 · Etiquetado débil con un LLM (Gemini)
**Objetivo:** que un LLM etiquete todos los titulares. Esas etiquetas servirán para **entrenar** los modelos.

**Qué hace:** la misma lógica de *lab9* y *lecture-finetuning* (system prompt con las 4 categorías, delimitador
`####` y un ejemplo de respuesta en JSON), pero con **Gemini** (`gemini-2.5-flash`) de Google AI Studio en lugar de
OpenAI, porque tiene una cuota gratuita. `temperature=0` para que sea reproducible. Los titulares se envían en lotes de 25.

**Clave de API:** se lee del archivo `.env` (`GEMINI_API_KEY=...`), que está en `.gitignore` y **no se sube a GitHub**.

**Resultado esperado:** la estimación de tokens (4a) y `datos/etiquetas_llm.csv` con una etiqueta y una confianza por
titular (4b).

In [5]:
# 4a) Prompt y conteo de tokens
import tiktoken

MODELO_LLM = "gemini-flash-latest"
RUTA_LLM = "datos/etiquetas_llm.csv"
TAM_LOTE = 25
DELIMITER = "####"

system_prompt = f"""
Eres un editor de un diario peruano que clasifica titulares de noticias.
Te daré varios titulares numerados, delimitados por {DELIMITER}.
Clasifica cada uno en EXACTAMENTE una de estas 4 etiquetas:

politica: Gobierno, Congreso, partidos, elecciones, justicia y fiscalía vinculadas a autoridades, política internacional.
economia: mercados, empresas, empleo, precios, impuestos, finanzas personales, comercio, minería como actividad económica.
deportes: fútbol, vóley, atletismo, competencias, clubes, deportistas.
ciencia_tecnologia: investigación científica, salud basada en estudios, medio ambiente, espacio, inteligencia artificial, tecnología.

Si un titular mezcla temas, elige el tema PRINCIPAL. "confianza" es tu probabilidad (0 a 1) de acertar.
Responde SOLO con un JSON, como en este ejemplo:

{DELIMITER}
0. Congreso aprueba nuevo retiro de fondos de AFP
1. Alianza Lima vence 2-0 a Cristal y lidera el Clausura
{DELIMITER}

{{"resultados": [{{"id": 0, "categoria": "politica", "confianza": 0.7}},
                 {{"id": 1, "categoria": "deportes", "confianza": 0.99}}]}}
"""

def mensaje_usuario(lote):
    """Arma el mensaje del usuario: titulares numerados entre delimitadores."""
    return DELIMITER + "\n" + "\n".join(f"{i}. {t}" for i, t in lote) + "\n" + DELIMITER

# Clase 1: gpt-4o-mini usa el tokenizador o200k_base
enc = tiktoken.get_encoding("o200k_base")
ejemplo_lote = list(enumerate(df["titular"].head(TAM_LOTE)))
tok_lote = len(enc.encode(system_prompt)) + len(enc.encode(mensaje_usuario(ejemplo_lote)))
n_lotes = -(-len(df) // TAM_LOTE)                       # división redondeando hacia arriba
TOKENS_ENTRADA = tok_lote * n_lotes
print(f"≈ {tok_lote} tokens por lote × {n_lotes} lotes = ≈ {TOKENS_ENTRADA:,} tokens de entrada en total")

≈ 929 tokens por lote × 18 lotes = ≈ 16,722 tokens de entrada en total


In [6]:
# 4b) Etiquetar con Gemini (guarda en cada lote; si se corta, vuelve a ejecutar y continúa)
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv(override=True)                               # lee GEMINI_API_KEY del archivo .env
if not os.environ.get("GEMINI_API_KEY"):
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Ingresa tu GEMINI_API_KEY: ").strip()
cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def etiquetar_lote(lote):
    respuesta = cliente.models.generate_content(
        model=MODELO_LLM,
        contents=mensaje_usuario(lote),                  # mensaje del usuario: los titulares
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,            # equivale al "system" de chat.completions
            temperature=0,
            response_mime_type="application/json",       # obliga a responder en JSON
        ),
    )
    datos = json.loads(respuesta.text)
    datos = datos["resultados"] if isinstance(datos, dict) else datos
    return {int(d["id"]): (d["categoria"], float(d["confianza"])) for d in datos}

llm = (pd.read_csv(RUTA_LLM, encoding="utf-8-sig") if os.path.exists(RUTA_LLM)
       else pd.DataFrame(columns=["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha"]))
llm = llm.dropna(subset=["etiqueta_llm"])                # descarta intentos fallidos anteriores
pendientes = [t for t in df["titular"] if t not in set(llm["titular"])]
print("Titulares por etiquetar:", len(pendientes))

t0 = time.time()
for k in range(0, len(pendientes), TAM_LOTE):
    lote = list(enumerate(pendientes[k:k + TAM_LOTE]))
    res = {}
    for intento in range(3):                             # hasta 3 intentos por lote
        try:
            res = etiquetar_lote(lote)
            break
        except Exception as e:
            print("  reintento:", type(e).__name__, str(e)[:150])
            time.sleep(20 * (intento + 1))               # la cuota gratis limita solicitudes por minuto
    nuevas = [{"titular": t,
               "etiqueta_llm": res[i][0] if i in res else None,
               "confianza": res[i][1] if i in res else None,
               "modelo_llm": MODELO_LLM, "fecha": date.today().isoformat()} for i, t in lote]
    llm = pd.concat([llm, pd.DataFrame(nuevas)], ignore_index=True)
    llm.to_csv(RUTA_LLM, index=False, encoding="utf-8-sig")             # guarda en cada lote
    print(f"  {min(k + TAM_LOTE, len(pendientes))}/{len(pendientes)}")
    time.sleep(7)                                        # respeta el límite de la cuota gratuita
LAT_LLM = (time.time() - t0) / len(pendientes) if pendientes else None

llm = llm.dropna(subset=["etiqueta_llm"])
llm = llm[llm["etiqueta_llm"].isin(CATEGORIAS)].drop_duplicates("titular")
llm.to_csv(RUTA_LLM, index=False, encoding="utf-8-sig")
print("Etiquetas válidas:", len(llm), "de", len(df))
llm.head()

Titulares por etiquetar: 58
  reintento: ServerError 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 
  25/58
  50/58
  58/58
Etiquetas válidas: 433 de 433


,titular,etiqueta_llm,confianza,modelo_llm,fecha
0,Comisión de Constitución aún no convoca a sesi...,politica,0.99,gemini-flash-latest,2026-10-18
1,Debate de pedido de facultades sigue entrampad...,politica,0.99,gemini-flash-latest,2026-10-18
2,Rosa Bartra asegura que la bicameralidad está ...,politica,0.98,gemini-flash-latest,2026-10-18
3,Fiscalía pide crédito suplementario al Congres...,politica,0.92,gemini-flash-latest,2026-10-18
4,Solicitan la destitución de Roberto Burneo por...,politica,0.98,gemini-flash-latest,2026-10-18


**Resultado:** el LLM etiquetó los **433 titulares** (`datos/etiquetas_llm.csv`), con su confianza. El etiquetado se
hizo en dos sesiones por los límites de la cuota gratuita: la API respondió a veces `503` (alta demanda) y `429`
(cuota diaria agotada). Gracias a que el código guarda cada lote y reintenta, se completó sin repetir trabajo.

In [7]:
# 5a) Unión de etiquetas
df = df.drop(columns=["etiqueta_llm", "confianza"], errors="ignore")   # permite re-ejecutar la celda
n_antes = len(df)
df = df.merge(llm[["titular", "etiqueta_llm", "confianza"]], on="titular", how="left", validate="one_to_one")
assert len(df) == n_antes, f"El merge cambió el número de filas: {n_antes} → {len(df)}"
print(f"Cruce OK: {len(df)} filas | sin etiqueta LLM: {df['etiqueta_llm'].isna().sum()}")
print(f"Coincidencia LLM vs sección del diario: {(df['etiqueta_llm'] == df['etiqueta']).mean():.1%}")
pd.crosstab(df["etiqueta"], df["etiqueta_llm"], margins=True,
            rownames=["Sección del diario"], colnames=["Etiqueta del LLM"])

Cruce OK: 433 filas | sin etiqueta LLM: 0
Coincidencia LLM vs sección del diario: 93.5%


Etiqueta del LLM,ciencia_tecnologia,deportes,economia,politica,All
Sección del diario,,,,,
ciencia_tecnologia,90,0,4,6,100
deportes,1,109,0,0,110
economia,6,0,96,6,108
politica,0,0,5,110,115
All,97,109,105,122,433


**Resultado:** el cruce no perdió ni ganó filas (433) y todas tienen etiqueta del LLM. El LLM coincide con la sección
del diario en el **93.5 %** de los casos. **Deportes** es casi perfecto (109 de 110). Los desacuerdos se concentran entre
**economía y política** (11 casos en total: Congreso, impuestos, AFP, presupuesto) y entre **economía y ciencia y
tecnología** (10: empresas tecnológicas, IA en negocios). Además, 6 notas de la sección de ciencia el LLM las considera
política. No siempre es un error del LLM: una nota de la sección "economía" puede tratar sobre todo de un debate en el
Congreso. Por eso la auditoría humana es necesaria.

In [9]:
# Auditoría: se muestran solo titular y bajada, para decidir sin ver la etiqueta del LLM
aud = pd.read_csv("datos/auditoria_20.csv", encoding="utf-8-sig")
for i, fila in aud.iterrows():
    print(f"{i:>2}. {fila['titular']}")
    print(f"     ↳ {str(fila['bajada'])[:160]}\n")

 0. El Nobel de Física atribuye su galardón al talento de sus colaboradores y la suerte
     ↳ Antes de iniciar el proyecto “todo el mundo pensaba que quizás era una buena idea, pero muy pocos creían que funcionaría, incluido yo mismo”, admitió Halzen.

 1. AlaCena no tiene precio
     ↳ “La propia compañía explica que las marcas compradas pueden reconocerse como activos, pero las creadas internamente no”, escribe Paul Thorndike, empresario y mi

 2. “Disculpas por los errores”: Gianni Infantino pidió perdón por el fallido plan de vender el Mundial
     ↳ Infantino se disculpa ante el Consejo de la FIFA por su proyecto de privatización de la Copa del Mundo.

 3. Previa Alianza Lima vs Atlético Grau EN VIVO: a qué hora y dónde ver el partido de hoy por el Torneo Clausura de la Liga 1 2026
     ↳ El enfrentamiento por la jornada 12 del Torneo Clausura entre los equipos de Alianza Lima y Atlético Grau se disputará en el estadio Alejandro Villanueva.

 4. León XIV fue operado de cáncer de 

In [18]:
# Mi etiqueta para cada titular, en el orden mostrado arriba (0 a 19).
# Opciones: "politica", "economia", "deportes", "ciencia_tecnologia"
ETIQUETAS_HUMANAS = [
    "ciencia_tecnologia",   # 0
    "economia",   # 1
    "deportes",   # 2
    "deportes",   # 3
    "ciencia_tecnologia",   # 4 Caso dudoso: “León XIV fue operado de cáncer de pulmón…”. El titular gira en torno a una figura pública, pero la nota es de divulgación médica (especialistas explican el procedimiento) y el diario la publica en Ciencias. Se etiquetó como ciencia_tecnologia.
    "ciencia_tecnologia",   # 5
    "politica",   # 6
    "deportes",   # 7
    "deportes",   # 8
    "ciencia_tecnologia",   # 9
    "deportes",   # 10
    "politica",   # 11
    "ciencia_tecnologia",   # 12
    "deportes",   # 13
    "deportes",   # 14
    "ciencia_tecnologia",   # 15
    "politica",   # 16
    "economia",   # 17
    "politica",   # 18
    "politica",   # 19
]
assert len(ETIQUETAS_HUMANAS) == 20
aud["etiqueta_humana"] = ETIQUETAS_HUMANAS
aud.to_csv("datos/auditoria_20.csv", index=False, encoding="utf-8-sig")
print("Guardado")

Guardado


In [19]:
# 5b) Auditoría humana de 20 casos
RUTA_AUD = "datos/auditoria_20.csv"
if not os.path.exists(RUTA_AUD):
    aud = df.dropna(subset=["etiqueta_llm"]).sample(20, random_state=SEMILLA)[["titular", "bajada", "etiqueta_llm", "confianza"]]
    aud["etiqueta_humana"] = ""
    aud.to_csv(RUTA_AUD, index=False, encoding="utf-8-sig")
    print("Se creó", RUTA_AUD, "→ complétalo y vuelve a ejecutar esta celda.")

aud = pd.read_csv(RUTA_AUD, encoding="utf-8-sig").fillna({"etiqueta_humana": ""})
aud["etiqueta_humana"] = aud["etiqueta_humana"].str.strip()
completas = aud["etiqueta_humana"].isin(CATEGORIAS)
if completas.all():
    aciertos = int((aud["etiqueta_humana"] == aud["etiqueta_llm"]).sum())
    ACUERDO_LLM_HUMANO = f"{aciertos}/20"
    print(f"Acuerdo LLM–humano: {aciertos}/20 = {aciertos / 20:.0%}")
    print("Casos en desacuerdo:")
    display(aud[aud["etiqueta_humana"] != aud["etiqueta_llm"]][["titular", "etiqueta_llm", "etiqueta_humana", "confianza"]])
else:
    ACUERDO_LLM_HUMANO = None
    print(f"Faltan {int((~completas).sum())} filas por completar en {RUTA_AUD}")

Acuerdo LLM–humano: 19/20 = 95%
Casos en desacuerdo:


,titular,etiqueta_llm,etiqueta_humana,confianza
6,Unión Europea y Minem acuerdan creación de mes...,economia,politica,0.9


**Comentario de la auditoría:** el acuerdo fue de **19/20 (95 %)**. Según el criterio del profesor en la clase 2
(si el LLM falla en ~5 %, sus etiquetas son confiables; si falla en 20–30 %, hay que mejorar el prompt), **las
etiquetas del LLM son confiables para entrenar el modelo**.

El único desacuerdo fue un caso límite: *"Unión Europea y Minem acuerdan creación de mesa técnica para destrabar
permisos e infraestructura"*. El LLM la etiquetó como **economía** (confianza 0.90), porque el objetivo es destrabar
proyectos de inversión. Yo la etiqueté como **política**, porque el actor principal es un ministerio y el diario la
publica en *Política › Gobierno*. No es un error claro del LLM, sino una noticia que mezcla ambos temas.

Otro caso dudoso fue *"León XIV fue operado de cáncer de pulmón…"*: el titular gira en torno a una figura pública,
pero la nota es divulgación médica (especialistas explican el procedimiento) y el diario la publica en *Ciencias*.
Coincidí con el LLM en **ciencia y tecnología**.